# Lilly — train the listening on real Bosnian speech

Set these in the panel on the right:

- **Session options → Accelerator → GPU** (P100 or T4)
- **Session options → Internet → On**

Nothing to upload — this notebook fetches its own speech.

Then **Save Version → Save & Run All (Commit)** and close the tab.

It measures **before** and **after** on the same held-out clips. That comparison is the
whole point: a run that does not lower the error rate did not work, however cleanly it
finished. Every step stops the run if it fails.


In [ ]:
# 1. Stop here unless the machine is actually set up
# Kaggle marks a version "complete" whenever no cell RAISES — a shell command that
# fails is not enough. So every check below is Python, and every later step is a
# checked subprocess. A misconfigured run should die in ten seconds, not in ten hours.
import os, subprocess, sys, urllib.error, urllib.request
from pathlib import Path

import torch
assert torch.cuda.is_available(), (
    "No GPU. Right panel -> Session options -> Accelerator -> GPU, then Save & Run All again.")

# Pin to one GPU on purpose. With two visible, the trainer splits each batch across
# both, which doubles the effective batch and halves the optimizer steps while the
# learning rate stays put — a different recipe than the one these numbers were set for.
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
print(torch.cuda.device_count(), "GPU(s) visible, using:", torch.cuda.get_device_name(0))

def reachable(url):
    try:
        urllib.request.urlopen(url, timeout=20).close()
    except urllib.error.HTTPError:
        pass          # a status code still proves we got out
    except Exception as exc:
        raise SystemExit(
            f"Cannot reach {url} ({exc}). Right panel -> Session options -> Internet -> On.")

for host in ("https://github.com", "https://pypi.org", "https://huggingface.co", "https://datasets-server.huggingface.co"):
    reachable(host)
print("network ok")

def run(*cmd):
    """Run a step and let a failure actually stop the notebook."""
    print("$", " ".join(str(c) for c in cmd), flush=True)
    subprocess.run([str(c) for c in cmd], check=True)


In [ ]:
# 2. Get the Lilly code
os.chdir("/kaggle/working")
subprocess.run(["rm", "-rf", "Lilly"], check=True)
run("git", "clone", "-q", "https://github.com/ssaaffaakk/Lilly.git")
assert Path("/kaggle/working/Lilly/training").is_dir(), "clone produced nothing"
os.chdir("/kaggle/working/Lilly")
print("working in", os.getcwd())


In [ ]:
# 3. Install what we need (~3 min)
# The versions are read out of the repo's own requirements.txt rather than
# copied into this cell. A second, hand-kept list is exactly how the last run
# died: peft is pinned in requirements.txt and was simply absent from here, so
# Kaggle's own much newer peft got used instead — and that one's torchao
# dispatcher raises against the torchao Kaggle also ships. The first
# get_peft_model() call blew up, after a 3 GB download and forty minutes.
NEEDED = ["transformers", "accelerate", "peft", "faster-whisper", "ctranslate2",
          "soundfile", "scipy", "pyarrow"]
pins = {}
for line in Path("requirements.txt").read_text(encoding="utf-8").splitlines():
    line = line.split("#")[0].strip()
    if "==" in line:
        pins[line.split("==")[0].strip().lower()] = line
unpinned = [n for n in NEEDED if n not in pins]
print("pinned here:", [pins[n] for n in NEEDED if n in pins])
print("no pin in requirements.txt, taking latest:", unpinned or "none")
run(sys.executable, "-m", "pip", "install", "-q", *[pins.get(n, n) for n in NEEDED])


In [ ]:
# 3b. Prove this machine can actually train, before an hour is spent finding out
# Two things have to hold and neither shows up in a version number: peft must be
# able to build a LoRA layer on this image, and the GPU Kaggle handed us must
# actually run kernels. The last run satisfied "GPU is available" and still could
# not compute — Kaggle gave a P100 (sm_60) that the installed PyTorch does not
# support, and separately peft could not build a layer at all.
#
# So: build a real LoRA layer, put it on the GPU, push a gradient through it. It
# takes about twenty seconds and it fails here, loudly, instead of after the
# download.
import torch, torch.nn as nn
from peft import LoraConfig, get_peft_model
import peft, transformers
print("peft", peft.__version__, "| transformers", transformers.__version__)

class Tiny(nn.Module):
    def __init__(self):
        super().__init__()
        self.q_proj = nn.Linear(32, 32)
    def forward(self, x):
        return self.q_proj(x)

tiny = get_peft_model(Tiny(), LoraConfig(r=4, target_modules=["q_proj"]))
try:
    tiny = tiny.cuda()
    out = tiny(torch.randn(4, 32, device="cuda")).sum()
    out.backward()
except RuntimeError as exc:
    raise SystemExit(
        f"The GPU cannot run this build of PyTorch ({exc}).\n"
        f"Card: {torch.cuda.get_device_name(0)}. Right panel -> Session options ->"
        f" Accelerator -> GPU T4 x2, then Save & Run All again.") from exc

lora = [n for n, p in tiny.named_parameters() if "lora_" in n and p.grad is not None]
assert lora, "peft built a LoRA layer but no gradient reached it"
print(f"LoRA trains on {torch.cuda.get_device_name(0)}: "
      f"{len(lora)} adapter tensors took a gradient")
del tiny
torch.cuda.empty_cache()


In [ ]:
# 4. Download the speech: clips to train on, and clips held back to judge with (~3 GB)
# The audio goes to scratch space, not into /kaggle/working — everything in the working
# directory is copied into the version Output, and 3 GB of wav files there is waste.
scratch = Path("/kaggle/temp/speech" if Path("/kaggle/temp").is_dir() else "/tmp/lilly-speech")
scratch.mkdir(parents=True, exist_ok=True)
if Path("data/speech").is_symlink():
    Path("data/speech").unlink()          # re-running the cell should not fail
if not Path("data/speech").exists():
    Path("data/speech").symlink_to(scratch)

run("python3", "data/scripts/download_speech_data.py")

for split in ("train", "valid", "test"):
    n = sum(1 for _ in open(f"data/speech/{split}.tsv", encoding="utf-8"))
    assert n > 100, f"{split}.tsv has only {n} clips — a download failed"
    print(f"{split}: {n:,} clips")


In [ ]:
# 5. BEFORE: how bad is the untrained listener on Bosnian it has never heard?
# Same format, same measure as the after-run, so the two numbers are comparable.
run("python3", "training/train_speech.py", "--base", "openai/whisper-small",
    "--convert-only", "/kaggle/working/listen-before")
run("python3", "training/evaluate_speech.py", "--data", "data/speech/test.tsv",
    "--model", "/kaggle/working/listen-before", "--limit", "200", "--show", "3")


In [ ]:
# 6. THE REAL TRAINING (~1-2 hours)
# --no-convert on purpose: the trained checkpoint is saved and packaged in the next
# cell before anything else can go wrong with it.
run("python3", "training/train_speech.py", "--data", "data/speech/train.tsv", "--no-convert")
assert Path("models/lilly/listen-trained/model.safetensors").is_file(), "nothing was trained"


In [ ]:
# 7. Convert the trained checkpoint into the format the app loads
run("python3", "training/train_speech.py",
    "--base", "models/lilly/listen-trained", "--convert-only", "models/lilly/listen")
assert Path("models/lilly/listen/model.bin").is_file(), "conversion produced nothing"


In [ ]:
# 8. AFTER: the same clips, the same measure. Lower is better.
run("python3", "training/evaluate_speech.py", "--data", "data/speech/test.tsv",
    "--limit", "200", "--show", "3")


In [ ]:
# 9. Package both listeners — the trained one, and the untrained one to fall back to
run("zip", "-qr", "/kaggle/working/lilly-listen.zip", "models/lilly/listen")
size = Path("/kaggle/working/lilly-listen.zip").stat().st_size
assert size > 1_000_000, f"the zip is only {size} bytes"
print(f"lilly-listen.zip — {size / 1048576:.0f} MB")
print("listen-before/ is in the Output too, if you need to go back")


**Done.** Compare the error rates printed by cell 5 and cell 8 — same clips, same
measure, so the difference is real.

If it dropped, download `lilly-listen.zip` from the **Output** tab and unzip it over
`models/lilly/listen/`. Keep a copy of the listener you are replacing first; the app has
no other way back.

If it did not drop, keep what you have. More clips, or more epochs, before another run.
